In [1]:
import os
import torch
import pytorch_lightning as pl
from transformers import get_scheduler, AutoModelForCausalLM, AutoProcessor, AutoConfig
from florence2_large import processing_florence2
from peft import LoraConfig, get_peft_model, PeftModel, PeftConfig
from pytorch_lightning import Trainer
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import MultiLabelBinarizer
from checkpoint_callback import CustomModelCheckpoint
from sklearn.model_selection import train_test_split
from torchvision.transforms.functional import to_pil_image
import pandas as pd
import numpy as np
import ast
import torchvision.transforms as T
import supervision as sv
import cv2
from PIL import Image
import yaml
import pydicom
from pydicom.pixel_data_handlers.util import apply_voi_lut
from difflib import get_close_matches

# os.environ["TOKENIZERS_PARALLELISM"] = "false"

# %env PYTORCH_NO_CUDA_MEMORY_CACHING=1

C:\Users\Mark\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
torch.cuda.empty_cache()

In [3]:
def load_config(config_path):
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    return config

In [4]:
CLASSES = ['No finding', 'Pleural thickening', 'Aortic enlargement', 'Pulmonary fibrosis', 'Cardiomegaly', 'Nodule or Mass', 'Lung Opacity', 'Other lesion', 'Pleural effusion', 'ILD', 'Infiltration', 'Calcification', 'Consolidation', 'Atelectasis', 'Rib fracture', 'Mediastinal shift', 'Enlarged PA', 'Pneumothorax', 'Emphysema', 'Lung cavity', 'Lung cyst', 'Clavicle fracture', 'Edema']
CLASSES = [cls.lower() for cls in CLASSES]

In [5]:
# loads the cofig for running the model
config_path = "configs/experiment.yaml"
config = load_config(config_path)

In [6]:
# uses GPU if able
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# DEVICE = torch.device("cpu")
# DEVICE = torch.device("xpu")

# state version of model if needed
# REVISION = 

MODEL_NAME = "microsoft/Florence-2-large"

### initialising the model
# downloads the model config from hugging face 
config_model = AutoConfig.from_pretrained(MODEL_NAME, trust_remote_code=True)
config_model.vision_config.model_type = "davit"
# model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, trust_remote_code = True, config = config_model,revision = REVISION).to(DEVICE)
# builds generic model using florence2
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, trust_remote_code = True, config = config_model).to(DEVICE)
# defines the processor to be used (florence2)
processor = processing_florence2.Florence2Processor.from_pretrained("./florence2_large")
processor.image_processor.size = config['model']['processor']['image_size']
processor.image_processor.crop_size = config['model']['processor']['crop_size']

In [7]:
def evaluate_results(model, inputs,processor, answers, images, batch_idx, questions):
    # import pdb;pdb.set_trace()
    bounding_box_annotator = sv.BoxAnnotator(color_lookup=sv.ColorLookup.INDEX)
    label_annotator = sv.LabelAnnotator(color_lookup=sv.ColorLookup.INDEX)
    color_annotator = sv.ColorAnnotator(color_lookup=sv.ColorLookup.INDEX)
    
    generated_ids = model.generate(input_ids=inputs["input_ids"],pixel_values=inputs["pixel_values"],max_new_tokens=300,num_beams=1)
    
    # model.generate(input_ids=inputs["input_ids"],pixel_values=inputs["pixel_values"],max_new_tokens=300,  num_beams=3)
    
    # Decode the predicted answers
    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)

    targets = []
    predictions = []
    img_list = []
    pred_text_list = []
    gt_text_list = []
    pred_label = []
    gt_label = []
    captions = []
    for i, text in enumerate(generated_text):
        answer = processor.post_process_generation(text, task='<CAPTION_TO_PHRASE_GROUNDING>', image_size=images[i].shape[:2])
        gt_answer = processor.post_process_generation(answers[i], task='<CAPTION_TO_PHRASE_GROUNDING>', image_size=images[i].shape[:2])
        
        # Create detections for both the ground truth and predicted answers
        # import pdb;pdb.set_trace()
        gt = sv.Detections.from_lmm(sv.LMM.FLORENCE_2, gt_answer, resolution_wh=images[i].shape)
        gt.class_id = np.array([CLASSES.index(class_name) for class_name in gt['class_name']])
        # import pdb;pdb.set_trace()
        class_label = gt['class_name'][0]

        # converts the image from a Tensor to a PIL image and then to a numpy array
        if isinstance(images[i], torch.Tensor):
            pil_image = to_pil_image(images[i].clone())
            cv_image = np.array(pil_image)
            image_with_ground_truth = bounding_box_annotator.annotate(pil_image, gt)
        else:
            cv_image = images[i]

        image_with_ground_truth = bounding_box_annotator.annotate(cv_image, gt)
        image_with_ground_truth = label_annotator.annotate(image_with_ground_truth, gt)
        pred_text_list.append(answer)
        gt_text_list.append(gt_answer)
        # if gt['class_name'][0] == 'infiltration':
        #     import pdb;pdb.set_trace()
        
        prediction = sv.Detections.from_lmm(sv.LMM.FLORENCE_2, answer, resolution_wh=images[i].shape)
        if prediction['class_name'] is not None and len(prediction['class_name']) > 0:
            prediction = prediction[~np.char.startswith(prediction['class_name'], 'mark')] 
            
        
            corrected_class_names = []
            # import pdb;pdb.set_trace()
            for class_name in prediction['class_name']:
                matches = get_close_matches(class_name, CLASSES, n=1, cutoff=0.5) 
                corrected_class_names.append(matches[0] if matches else class_name)
            prediction['class_name'] = corrected_class_names 
            prediction = prediction[np.isin(prediction['class_name'], CLASSES)] 
            
            prediction.class_id = np.array([CLASSES.index(class_name) for class_name in prediction['class_name']])
            prediction.confidence = np.ones(len(prediction))
            # import pdb;pdb.set_trace()
           
            per_sample_pred_cls = prediction['class_name'].tolist()
            per_sample_gt_cls = gt['class_name'].tolist()
            pred_label.append(per_sample_pred_cls)
            gt_label.append(per_sample_gt_cls)

            targets.append(gt)
            predictions.append(prediction)
            if i<10:
                prediction['class_name'] = ['pred_'+ class_label]*len(prediction['class_name']) 
                image_with_predictions = bounding_box_annotator.annotate(image_with_ground_truth.copy(), prediction)
                image_with_predictions = color_annotator.annotate(image_with_predictions, prediction)
                # image_with_predictions = label_annotator.annotate(image_with_predictions, prediction)
                # image_with_ground_truth = Image.fromarray(image_with_ground_truth.astype(np.uint8))
                image_with_predictions = Image.fromarray(image_with_predictions.astype(np.uint8))
                # res = combine_images(image_with_ground_truth, image_with_predictions)
                # img_list.append(res)
                img_list.append(image_with_predictions)
                captions.append(f"Question: {questions[i]}\nGround truth: {gt_answer}\nPredicted: {answer}")
        else:
            # import pdb;pdb.set_trace()
            # prediction.class_id = np.array([-1])
            targets.append(gt)
            predictions.append(prediction)
            if i<10:
                img_list.append(image_with_ground_truth)
                captions.append(f"Question: {questions[i]}\nGround truth: {gt_answer}\nPredicted: {answer}")
            pred_label.append([])
            gt_label.append(gt['class_name'])

    return {
        "res_samples": img_list,  
        "predictions": predictions, 
        "targets": targets,        
        "text_pred_answer":pred_text_list, 
        "text_gt_answer":gt_text_list,
        "pred_label":pred_label,
        "gt_label":gt_label,
        "captions":captions
    }
    # mean_average_precision = sv.MeanAveragePrecision.from_detections(predictions=predictions,targets=targets)



        # print(confusion_matrix.matrix)

    #     # Ensure valid class names
    #     prediction['class_name'] = correct_class_names(prediction['class_name'], CLASSES)
    #     prediction = prediction[np.isin(prediction['class_name'], CLASSES)]

    #     # Assign class_id and confidence for the prediction
    #     prediction.class_id = np.array([CLASSES.index(class_name) for class_name in prediction['class_name']])
    #     prediction.confidence = np.ones(len(prediction))

    #     # Annotate images with ground truth and predictions
    #     image_with_predictions = annotate_image(images[i], prediction, bounding_box_annotator, label_annotator)
    #     image_with_ground_truth = annotate_image(images[i], gt, bounding_box_annotator, label_annotator)

    #     # Convert to PIL images for saving
    #     image_with_ground_truth = Image.fromarray(image_with_ground_truth.astype(np.uint8))
    #     image_with_predictions = Image.fromarray(image_with_predictions.astype(np.uint8))

    #     # Combine and save images
    #     combined_image = combine_images(image_with_ground_truth, image_with_predictions)
    #     combined_image.save(f"./combined_image_{batch_idx}_{i}.png")

    #     processed_predictions.append((image_with_ground_truth, image_with_predictions))  # You could also store metrics here

    # return processed_predictions
    # return 'ok'



In [8]:
annotations = pd.read_csv("E:/vinbigdata_xrays/vinbigdata/train_original.csv")

# drop all rows that contain 'no finding'
# no finding rows don't have bboxes
annotations = annotations.drop(annotations[annotations['class_name'] == 'No finding'].index)

print(annotations['class_name'].value_counts())
# df.word.value_counts()['myword']

# create data splits
train_data, rest_data = train_test_split(annotations, train_size=0.8, shuffle=False)
validation_data, test_data = train_test_split(rest_data, test_size=0.5, shuffle=False)

# add split column
train_data['split'] = 'train'
validation_data['split'] = 'validate'
test_data['split'] = 'test'

# combine and save
split_annotations = pd.concat([train_data, validation_data, test_data]).reset_index(drop=True)
split_annotations.to_csv('E:/vinbigdata_xrays/vinbigdata/train.csv', index=False)

class_name
Aortic enlargement    7162
Cardiomegaly          5427
Pleural thickening    4842
Pulmonary fibrosis    4655
Nodule/Mass           2580
Lung Opacity          2483
Pleural effusion      2476
Other lesion          2203
Infiltration          1247
ILD                   1000
Calcification          960
Consolidation          556
Atelectasis            279
Pneumothorax           226
Name: count, dtype: int64


In [9]:
# converts DICOM files to np arrays
# copied from https://www.kaggle.com/code/raddar/convert-dicom-to-np-array-the-correct-way
def read_xray(path, voi_lut = True, fix_monochrome = True, target_size=(128, 128)):
    dicom = pydicom.dcmread(path)

    # VOI LUT (if available by DICOM device) is used to transform raw DICOM data to "human-friendly" view
    if voi_lut:
        data = apply_voi_lut(dicom.pixel_array, dicom)
    else:
        data = dicom.pixel_array
               
    # depending on this value, X-ray may look inverted - fix that:
    if fix_monochrome and dicom.PhotometricInterpretation == "MONOCHROME1":
        data = np.amax(data) - data

    # normalize to [0, 255]
    data = data - np.min(data)
    data = data / np.max(data)
    data = (data * 255).astype(np.uint8)

    # add padding to make the image square
    h, w = data.shape
    if h != w:
        max_dim = max(h, w)
        padded_image = np.zeros((max_dim, max_dim), dtype=np.uint8)
        padded_image[(max_dim - h) // 2:(max_dim - h) // 2 + h,
                     (max_dim - w) // 2:(max_dim - w) // 2 + w] = data
        data = padded_image

    # resize to target size
    data = cv2.resize(data, target_size, interpolation=cv2.INTER_LINEAR)

    # add channel dimension (C=1)
    data = np.expand_dims(data, axis=0)

    return data

In [10]:
class VindrDataset(Dataset):
    def __init__(self, img_root, annotation_csv, split='train', data_pct=1.0, transform=None):
        self.img_root = img_root
        self.transform = transform
        
        # Check if split is valid
        if split not in ['train', 'test', 'validate']:
            raise ValueError(f"Invalid split: {split}. Expected one of ['train', 'test', 'validate'].")
        
        # Check if data_pct is valid
        if not (0 < data_pct <= 1):
            raise ValueError(f"data_pct should be in the range (0, 1], got {data_pct}")
        
        self.annotations = pd.read_csv(annotation_csv)
        
        if split == 'train':
            self.annotations = self.annotations[(self.annotations['split'] == 'train') | (self.annotations['split'] == 'validate')].reset_index(drop=True)
        else:
            self.annotations = self.annotations[self.annotations['split'] == split].reset_index(drop=True)

        if self.annotations.empty:
            raise ValueError(f"No data available for split: {split}")

        # Sample data based on data_pct
        if data_pct < 1.0:
            sampled_indices = np.random.choice(len(self.annotations), size=int(len(self.annotations) * data_pct), replace=False)
            self.annotations = self.annotations.iloc[sampled_indices].reset_index(drop=True)

        # grouping annotations by image_id
        # each image has multiple annotations by different radiologistis for different anatomical defects
        self.grouped_annotations = self.annotations.groupby('image_id')
        self.image_ids = list(self.grouped_annotations.groups.keys())

        print(f"Loaded {len(self.annotations)} samples for split: {split} with data_pct: {data_pct}")

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        # Load image
        img_id = self.annotations.iloc[idx]['image_id']
        img_path = os.path.join(self.img_root, f"{img_id}.dicom")
        # image = np.array(Image.open(img_path).convert("RGB"))
        image = read_xray(img_path)

        # get all annotations for this image
        image_annotations = self.grouped_annotations.get_group(img_id)

        # extract bounding boxes and class names for this image
        boxes = []
        class_names = []
        for _, row in image_annotations.iterrows():
            boxes.append([row['x_min'], row['y_min'], row['x_max'], row['y_max']])
            class_name = row['class_name'].replace('vindrcxr/', '')
            if class_name == 'Nodule/Mass':
                class_name = 'Nodule or Mass'
            class_name = class_name.lower()
            class_names.append(class_name)
            
        print(f"Image {img_id} has {len(boxes)} bounding boxes.")

        # Convert grayscale (1 channel) to 3 channels
        if image.shape[0] == 1:  # Check if single-channel
            image = np.repeat(image, 3, axis=0)  # Repeat channel 3 times

        # makes the image a torch tensor
        if not isinstance(image, torch.Tensor):
            image = torch.tensor(image, dtype=torch.float32)
        
        # Assuming you want to return a single class name if all are the same,
        # otherwise, you might need to adjust how you handle multiple class names.
        # For now, let's just take the first class name.
        label = class_names[0] if class_names else 'no finding'  # Default to 'no finding' if no boxes
        
        return {
            'image': image,
            'boxes': boxes,  # List of lists
            'label': label  # Single class name for the image
        }
    

class DetInstructDataset_vindr(Dataset):
    # def __init__(self, base_dataset, scale_factor=1000, task="<OPEN_VOCABULARY_DETECTION>",task_prompt="Locate {input} in the image.", max_classes=5, max_invalid_cls=2):
    def __init__(self, base_dataset, task="<CAPTION_TO_PHRASE_GROUNDING>", task_prompt="Locate the phrases in the caption: {input}.", use_definition=True):
        self.base_dataset = base_dataset
        self.task_prompt = task_prompt
        self.task = task
        self.scale_factor = 1000 
        self.definition = yaml.safe_load(open('configs/vindr_definition.yaml'))
        self.use_definition = use_definition
        print('❗ Use definition:', self.use_definition)


    def __len__(self):
        return len(self.base_dataset)

    # def normalize_coordinates(self, bbox, image_shape):
    #     x1, y1, x2, y2 = bbox
    #     h, w = image_shape[:2] # image shape (H, W, C)
    #     normalized_x1 = int((x1 / w) * self.scale_factor)
    #     normalized_y1 = int((y1 / h) * self.scale_factor)
    #     normalized_x2 = int((x2 / w) * self.scale_factor)
    #     normalized_y2 = int((y2 / h) * self.scale_factor)
    #     return f"<loc_{normalized_x1}><loc_{normalized_y1}><loc_{normalized_x2}><loc_{normalized_y2}>"

    def normalize_coordinates(self, bbox, image_shape):
        if not isinstance(bbox, (list, tuple)) or len(bbox) != 4:
            raise ValueError(f"Invalid bounding box: {bbox}. Expected a list or tuple of four values.")
        x1, y1, x2, y2 = bbox
        h, w = image_shape[:2]  # image shape (H, W, C)
        normalized_x1 = int((x1 / w) * self.scale_factor)
        normalized_y1 = int((y1 / h) * self.scale_factor)
        normalized_x2 = int((x2 / w) * self.scale_factor)
        normalized_y2 = int((y2 / h) * self.scale_factor)
        return f"<loc_{normalized_x1}><loc_{normalized_y1}><loc_{normalized_x2}><loc_{normalized_y2}>"


    def __getitem__(self, idx):
        # Get data from the base dataset
        sample = self.base_dataset[idx]
        image = sample['image']
        bounding_boxes = sample['boxes']
        det_obj = sample['label']
        definition = self.definition[det_obj]
        answer = []
        for bbox in bounding_boxes:
            if len(bbox) != 4:
                raise ValueError(f"Bounding box {bbox} has invalid length: {len(bbox)}") # each xray has a set of bounding boxes
            locs = self.normalize_coordinates(bbox, image.shape)
            answer.append(f"{det_obj}{locs}")
        final_answer = "".join(answer)
        if self.use_definition:
                det_obj = '{} means {}.'.format(det_obj, definition)
        # Generate the task-specific prompt
        task_prompt = self.task_prompt.format(input=det_obj)
        task_prompt = self.task + task_prompt
        

        return {    
            'image': image,
            'question': task_prompt,
            'answer': final_answer,
            'task': self.task
        }



In [11]:
class VinderDataLoaderManager(pl.LightningDataModule):
    def __init__(self, config):
        super().__init__()
        # Extract parameters from config
        self.img_root = config.get("img_root")
        self.annotation_csv = config.get("annotation_csv")
        self.batch_size = config.get("batch_size", 8)
        self.data_pct = config.get("data_pct", 1.0)
        self.num_workers = config.get("num_workers", 0)
        self.device = config.get("device", torch.device("cuda" if torch.cuda.is_available() else "cpu"))
        # self.device = config.get("device", "cpu")
        # self.device = config.get("device", "xpu")
        self.use_definition = config["use_definition"]
        
        # Use the passed processor or initialize a default one


    def collate_fn(self,batch):
        # # Unzip the batch into questions, answers, and images
        # questions = [item['question'] for item in batch]
        # answers = [item['answer'] for item in batch]
        # images = [item['image'] for item in batch]
        # tasks = [item['task'] for item in batch]
    
        # return images, questions, answers, tasks

        images = torch.stack([item['image'] for item in batch])  # Stack into a single tensor
        questions = [item['question'] for item in batch]
        answers = [item['answer'] for item in batch]
        tasks = [item['task'] for item in batch]
        return images, questions, answers, tasks
            
    
    def create_dataloader(self, split):
        """
        Creates a DataLoader for the given dataset split (train/val/test).
        
        Args:
            split (str): The dataset split ('train', 'val', or 'test').

        Returns:
            DataLoader: The DataLoader for the given split.
        """
        # Initialize the base dataset (MIMICDataset)
        base_dataset = VindrDataset(
            img_root=self.img_root,
            annotation_csv=self.annotation_csv,
            split=split,  # 'train', 'val', or 'test'
            data_pct=self.data_pct,
            transform=None
        )

        # Initialize the Multi_task_Instructer dataset
        # import pdb; pdb.set_trace()
        multi_task_dataset = DetInstructDataset_vindr(
            base_dataset=base_dataset,
            use_definition=self.use_definition
        )

        # Create DataLoader
        return DataLoader(
            multi_task_dataset,
            batch_size=self.batch_size,
            collate_fn=self.collate_fn,  # Use the custom collate function
            num_workers=self.num_workers,  # Adjust based on your system's capabilities
            shuffle=True if split == 'train' else False
        )
    
    def train_dataloader(self):
        """
        Returns the DataLoader for the training set.
        """
        return self.create_dataloader(split='train')

    def val_dataloader(self):
        """
        Returns the DataLoader for the validation set.
        """
        return self.create_dataloader(split='validation')

    def test_dataloader(self):
        """
        Returns the DataLoader for the test set.
        """
        return self.create_dataloader(split='test')


In [12]:
class FlorenceLightningModel(pl.LightningModule):
    def __init__(self, model, processor, lr=1e-6, num_training_steps=None):
        super(FlorenceLightningModel, self).__init__()
        self.model = model
        self.processor = processor
        self.lr = float(lr)
        self.num_training_steps = num_training_steps
        self.test_outputs = []
        self.valid_outputs = []

    def training_step(self, batch, batch_idx):
        images, questions, answers, tasks = batch
        inputs = self.processor(
            text=questions,
            images=list(images),  # Ensure images are passed as a list
            return_tensors="pt",
            padding=True,
            image_mean=[0.5, 0.5, 0.5],
            image_std=[0.5, 0.5, 0.5]
        ).to(self.device)
        
        input_ids = inputs["input_ids"]
        pixel_values = inputs["pixel_values"]
        labels = self.processor.tokenizer(
            text=answers,
            return_tensors="pt",
            padding=True,
            return_token_type_ids=False
        ).input_ids.to(self.device)

        outputs = self.model(input_ids=input_ids, pixel_values=pixel_values, labels=labels)
        loss = outputs.loss
        self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True, batch_size=len(images), sync_dist=True)
        torch.cuda.empty_cache()
        return loss

    def validation_step(self, batch, batch_idx):
        images, questions, answers, tasks = batch
        inputs = self.processor(
            text=questions,
            images=list(images),  # ensure images are passed as a list
            return_tensors="pt",
            padding=True,
            image_mean=[0.5, 0.5, 0.5],
            image_std=[0.5, 0.5, 0.5]
        ).to(self.device)
 
        batch_results = evaluate_results(
            model=self.model, 
            inputs=inputs,
            processor=self.processor, 
            answers=answers, 
            images=images,
            batch_idx=batch_idx,
            questions=questions
        )

        self.valid_outputs.append(batch_results)
        return batch_results

    def on_validation_epoch_end(self):
        all_predictions = []
        all_targets = []

        for batch_result in self.valid_outputs:
            all_predictions.extend(batch_result["predictions"])
            all_targets.extend(batch_result["targets"])

        confusion_matrix = sv.ConfusionMatrix.from_detections(
            predictions=all_predictions, 
            targets=all_targets, 
            classes=CLASSES
        )

        mean_average_precision = sv.MeanAveragePrecision.from_detections(
            predictions=all_predictions, 
            targets=all_targets
        )

        self.log("val/mAP_50_95", mean_average_precision.map50_95)
        self.log("val/mAP_50", mean_average_precision.map50)
        self.log("val/mAP_75", mean_average_precision.map75)

    def test_step(self, batch, batch_idx):
        images, questions, answers, tasks = batch
        inputs = self.processor(
            text=questions,
            images=list(images),  # Ensure images are passed as a list
            return_tensors="pt",
            padding=True,
            image_mean=[0.5, 0.5, 0.5],
            image_std=[0.5, 0.5, 0.5]
        ).to(self.device)

        batch_results = evaluate_results(
            model=self.model, 
            inputs=inputs,
            processor=self.processor, 
            answers=answers, 
            images=images,
            batch_idx=batch_idx,
            questions=questions
        )
        self.test_outputs.append(batch_results)
        return batch_results

    def on_test_epoch_end(self):
        all_predictions = []
        all_targets = []

        for batch_result in self.test_outputs:
            all_predictions.extend(batch_result["predictions"])
            all_targets.extend(batch_result["targets"])
        
        confusion_matrix = sv.ConfusionMatrix.from_detections(
            predictions=all_predictions, 
            targets=all_targets, 
            classes=CLASSES
        )

        mean_average_precision = sv.MeanAveragePrecision.from_detections(
            predictions=all_predictions, 
            targets=all_targets
        )

        print("mAP_50_95:", mean_average_precision.map50_95)
        print("mAP_50:", mean_average_precision.map50)
        print("mAP_75:", mean_average_precision.map75)
        self.log("test/mAP_50_95", mean_average_precision.map50_95)
        self.log("test/mAP_50", mean_average_precision.map50)
        self.log("test/mAP_75", mean_average_precision.map75)

        print("Confusion Matrix:\n", confusion_matrix.matrix)
        print("Mean Average Precision:\n", mean_average_precision)

    def configure_optimizers(self):
        print('self.lr:', self.lr)
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=self.lr)
        lr_scheduler = get_scheduler(
            name="linear",
            optimizer=optimizer,
            num_warmup_steps=0,
            num_training_steps=self.num_training_steps,
        )
        return [optimizer], [lr_scheduler]


In [13]:
# # loads the cofig for running the model
# config_path = "configs/experiment.yaml"
# config = load_config(config_path)

In [14]:
# # uses GPU if able
# DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# # state version of model if needed
# # REVISION = 

# MODEL_NAME = "microsoft/Florence-2-large"

# ### initialising the model
# # downloads the model config from hugging face 
# config_model = AutoConfig.from_pretrained(MODEL_NAME, trust_remote_code=True)
# config_model.vision_config.model_type = "davit"
# # model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, trust_remote_code = True, config = config_model,revision = REVISION).to(DEVICE)
# # builds generic model using florence2
# model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, trust_remote_code = True, config = config_model).to(DEVICE)
# # defines the processor to be used (florence2)
# processor = processing_florence2.Florence2Processor.from_pretrained("./florence2_large")
# processor.image_processor.size = config['model']['processor']['image_size']
# processor.image_processor.crop_size = config['model']['processor']['crop_size']

In [15]:
if config['model']['peft']['use_peft']:
    # load an existing peft model checkpoint if there is one
    if config['model']['peft']['lora_checkpoint'] not in [None, "False"]:
        lora_checkpoint = LoraConfig.from_pretrained(config['model']['peft']['lora_checkpoint'])
        model = PeftModel.from_pretrained(model, lora_checkpoint, is_trainable=True)
    else:
        lora_config = LoraConfig(
                r=8,
                lora_alpha=8,
                target_modules=["q_proj", "o_proj", "k_proj", "v_proj", "linear", "Conv2d", "lm_head", "fc2"],
                task_type="CAUSAL_LM",
                lora_dropout=0.05,
                bias="none",
                inference_mode=False,
                use_rslora=True,
                init_lora_weights="gaussian"
            )
        model = get_peft_model(model, lora_config)

# else, fine tune entire language part, only freeze the vision part
elif config['model']['finetune']:
    for param in model.vision_tower.parameters():
        param.requires_grad = False

# otherwise, train the entire model
else:
    for param in model.parameters():
        param.requires_grad = True

In [16]:
# config['dataset']['vindr']['data_pct'] = 1.0

data_loader_manager = VinderDataLoaderManager({
    "img_root": config['dataset']['vindr']['img_root'],
    "annotation_csv": config['dataset']['vindr']['annotation_csv'],
    "batch_size": config['trainer']['train_batch_size'],
    "data_pct": config['dataset']['vindr']['data_pct'],
    "num_workers": config['trainer']['num_workers'],
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    # "device": "xpu",
    # "device": "cpu",
    "processor": None,  # use default processor
    "use_definition": False  # change to True if want to use a definition as a prompt
})

In [17]:
train_dataloader = data_loader_manager.train_dataloader()
val_dataloader = data_loader_manager.test_dataloader()

Loaded 32486 samples for split: train with data_pct: 1.0
❗ Use definition: False
Loaded 3610 samples for split: test with data_pct: 1.0
❗ Use definition: False


In [18]:
dataset_size = len(train_dataloader.dataset)
num_training_steps = (dataset_size + config['trainer']['train_batch_size'] - 1) // config['trainer']['train_batch_size']

lightning_model = FlorenceLightningModel(model=model, processor=processor, lr=config['trainer']['learning_rate'], num_training_steps=num_training_steps)

In [19]:
if config['trainer']['checkpoint_dir'] is not None:
    os.makedirs(config['trainer']['checkpoint_dir'], exist_ok=True)


custom_checkpoint_callback = CustomModelCheckpoint(
    dirpath=config['trainer']['checkpoint_dir'],
    filename='model-{epoch}-{step}',
    save_top_k=1,  # Save top 2 models based on the monitored metric
    monitor='val/mAP_50_95',  # Monitor a different metric (e.g., val_accuracy)
    mode='min',  # Mode for monitoring (min for loss, max for accuracy)
    # every_n_train_steps=500,  # every_n_train_steps >= save_top_k*val_check_interval
    save_embedding_layers=True,  # Save the embedding layers
    verbose=True  # Set to True to log when checkpoints are saved
)

In [20]:
trainer = Trainer(
    max_epochs=config['trainer']['max_epochs'],
    # accelerator="xpu",
    # accelerator="cpu",
    accelerator="auto",
    # devices=1,
    devices="auto",
    strategy="auto",
    # log_every_n_steps=200,
    # logger=wandb_logger,
    num_sanity_val_steps=0,
    enable_checkpointing=True
    # callbacks=[custom_checkpoint_callback]
)

trainer.fit(lightning_model, train_dataloader, val_dataloader)

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 3060') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type                 | Params | Mode 
-------------------------------------------------------
0 | model | PeftModelForCausalLM | 833 M  | train
-------------------------------------------------------
4.1 M     Trainable params
828 M     Non-trainable params
833 M     Total params
3,332.476 Total estimated 

self.lr: 3e-06


Epoch 0:   0%|          | 0/6498 [00:00<?, ?it/s] Image 6d80520d7518642b92a089b028407104 has 7 bounding boxes.
Image 2e4b0a0fb7faf81bc42b46c3247ae8ad has 5 bounding boxes.
Image fa7b469e21cf7a4e7afe9f17cf721259 has 7 bounding boxes.
Image eca27ff9495044fbcd347ee51a8f1187 has 17 bounding boxes.
Image b2506496d79b479a6b1427288235190c has 8 bounding boxes.
Epoch 0:   0%|          | 1/6498 [00:04<7:36:31,  0.24it/s, v_num=69, train_loss_step=8.660]Image 7664d4a022c8ca9a94c0bb8adb48209c has 7 bounding boxes.
Image e1c7cdc2d2faec46612191158a77a38a has 13 bounding boxes.
Image 96856fd17a41d64ead800f556f369cdb has 5 bounding boxes.
Image fad39dc356aaa2da58470c6daaba8112 has 7 bounding boxes.
Image 6d80520d7518642b92a089b028407104 has 7 bounding boxes.
Epoch 0:   0%|          | 2/6498 [00:06<6:17:44,  0.29it/s, v_num=69, train_loss_step=7.260]Image dfb2b9c6d978d65bf5b78728f744c830 has 6 bounding boxes.
Image ae127045cca13da6dbfee11df1f08c7e has 15 bounding boxes.
Image fa109c087e46fe1ea27e48ce6

Token indices sequence length is longer than the specified maximum sequence length for this model (1247 > 1024). Running this sequence through the model will result in indexing errors


Image b78367d7be3c92afe88f13efcb6dcaf5 has 19 bounding boxes.
Epoch 0:   0%|          | 3/6498 [00:19<11:33:58,  0.16it/s, v_num=69, train_loss_step=12.00]Image 150d9b226334d9f3471d088307cdc5fd has 7 bounding boxes.
Image 1f897eadcbbca4e2104e7a2d7fd3e0f9 has 5 bounding boxes.
Image 22576c31ecae86e2e6d580b4bebb5d77 has 4 bounding boxes.
Image 1e1dcf1ea1d974a5fea81b7616a11723 has 8 bounding boxes.
Image d1ae8f8c681cf7e9ad6fc1f8ab02dc3a has 12 bounding boxes.
Epoch 0:   0%|          | 4/6498 [00:24<11:08:02,  0.16it/s, v_num=69, train_loss_step=7.670]Image 1caa3fd2d741bbb50cc3fdcc32b6f0cd has 20 bounding boxes.
Image a6df9baa059b2fd400ee2e0149ae5b87 has 3 bounding boxes.
Image 0211f9fe31142ba1e25fb6a2ea3fea38 has 7 bounding boxes.
Image 9ef084b09e407e4f8f00932602d1a88d has 8 bounding boxes.
Image 4c02da7cd2dc7415b226103114e5aaf0 has 14 bounding boxes.
Epoch 0:   0%|          | 5/6498 [00:29<10:27:57,  0.17it/s, v_num=69, train_loss_step=8.720]Image 566150c4fe08a31a5a17d9d9f6c5f21f has 8 b

...

Image ce2c5376b597eb067395da4c7f9960aa has 2 bounding boxes.
Image 0ea0bf774e0436aec7d1a1e62074c9cd has 2 bounding boxes.
Image 660d934ad8ecf6701af7eb703e1cb4c8 has 2 bounding boxes.
Image 1b5f927abb8fd8e0800d2ad31a5620c1 has 6 bounding boxes.
Image 330b3939ea1fe042ad347e8d02abc393 has 1 bounding boxes.


The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Image 25897fa85891d5fcf7c65d520628cf20 has 1 bounding boxes.
Image 2b24fc9d451749e2edfc5ff60be60192 has 1 bounding boxes.
Image 971e2965c2e5149f64be489d589dc3d0 has 2 bounding boxes.



Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined